In [47]:
#Importacion de librerias
import requests
from sqlalchemy import create_engine, text, inspect
from sqlalchemy.orm import sessionmaker
import pandas as pd
from unidecode import unidecode
import numpy as np
from tqdm import tqdm
import json
from pandas import json_normalize
import openpyxl
import concurrent.futures

#CREACION DE LA CONEXION CON BASE DE DATOS SQL SERVER

# Configura los detalles de la conexión
host='172.16.1.33'
port='1433'
database= 'CUN_REPOSITORIO'

try:
    # Crea la cadena de conexión
   #url = 'mssql+pyodbc://@{host}:{port}/{db}?driver=SQL+Server'.format(host=host, port=port, db=database)
   url = 'mssql+pyodbc://@{host}:{port}/{db}?trusted_connection=yes&driver=SQL+Server'.format(host=host, port=port, db=database)

   
    # Crear conexion con base de datos
   engine = create_engine(url) 
   print("Conexion a la base de datos realizada") 
    
except Exception as e:
    print('Error:',e)

Conexion a la base de datos realizada


In [2]:
with engine.connect() as connection:
    try:
        # 
        sql_query = text(''' 

        
        SELECT DISTINCT * FROM GRADOS.Consolidado_Grados
        WHERE NIVEL IN ('TECNICO','TECNOLOGO','PROFESIONAL')

 ''')

        # CONVIERTE RESULTADO DE CONSULTA EN DATAFRAME    
        df_grados = pd.read_sql_query(sql_query, engine)

    finally:
        
        connection.close()

In [3]:
# Asignar prioridad a los niveles educativos
prioridad_nivel = {'TECNICO': 1, 'TECNOLOGO': 2, 'PROFESIONAL': 3}
df_grados['prioridad'] = df_grados['NIVEL'].map(prioridad_nivel)

In [4]:
# Ordenar por identificación y prioridad descendente
df = df_grados.sort_values(by=['NUMERO', 'prioridad'], ascending=[True, False])

# Eliminar duplicados, dejando solo el último ciclo (el de mayor prioridad)
df_final = df.drop_duplicates(subset=['NUMERO'], keep='first')

In [5]:
df_final['NUMERO'].nunique()

82552

### ESPECIALIZACIONES PARA VER QUE ESTUDIANTES REALIZARON RECOMPRA

In [6]:
with engine.connect() as connection:
    try:
        # 
        sql_query = text(''' 

        
        SELECT DISTINCT NUMERO,NOM_PROGRAMA_PROFESIONAL AS NOM_DEPENDENCIA,[PLAN] as PENSUM_GRADO_ESP,[SNIES] as SNIES_GRADO_ESP
        FROM GRADOS.Consolidado_Grados
        WHERE NIVEL IN ('ESPECIALIZACION')

 ''')

        # CONVIERTE RESULTADO DE CONSULTA EN DATAFRAME    
        df_esp = pd.read_sql_query(sql_query, engine)

    finally:
        
        connection.close()

In [7]:
df_recompra_esp = pd.merge(df_final,df_esp,how='left',on=['NUMERO'])

In [8]:
df_recompra_esp['PENSUM_GRADO_ESP'] = df_recompra_esp['PENSUM_GRADO_ESP'].str.rstrip('ND')

### MALLAS PARA LLEVAR EL COD_UNIDAD

In [9]:
with engine.connect() as connection:
    try:
        # 
        sql_query = text(''' 

        
        SELECT DISTINCT COD_PROGRAMA AS COD_UNIDAD_MALLAS,SNIES AS SNIES_MALLAS,PENSUM AS PENSUM_MALLAS
        FROM CUN.MALLASCUN

 ''')

        # CONVIERTE RESULTADO DE CONSULTA EN DATAFRAME    
        df_mallas = pd.read_sql_query(sql_query, engine)

    finally:
        
        connection.close()

In [10]:
df_mallas[df_mallas['SNIES_MALLAS']=='106040']

,COD_UNIDAD_MALLAS,SNIES_MALLAS,PENSUM_MALLAS
1212,ESP08,106040,ESPPA
1226,ESP22,106040,ESPPA


In [11]:
df_mallas['PENSUM_MALLAS'] = df_mallas['PENSUM_MALLAS'].str.rstrip('ND')

In [12]:
df_grados_cod_uni = pd.merge(df_recompra_esp,df_mallas,how='left',left_on=['SNIES_GRADO_ESP','PENSUM_GRADO_ESP'],right_on=['SNIES_MALLAS','PENSUM_MALLAS'])

### ESTADISTICA PARA VER DIPLOMADOS Y ESPECIALIZACIONES MATRICULADAS

In [13]:
with engine.connect() as connection:
    try:
        # 
        sql_query = text(''' 

        
         SELECT DISTINCT NUM_IDENTIFICACION as NUMERO,NIVEL_FORMACION,NOM_DEPENDENCIA,COD_UNIDAD
        FROM CUN.ESTADISTICA_ESTUDIANTE_2

 ''')

        # CONVIERTE RESULTADO DE CONSULTA EN DATAFRAME    
        df_est = pd.read_sql_query(sql_query, engine)

    finally:
        
        connection.close()

#### SE CREA DF PARA SABER SI MATRICULARON Y NO ESTAN GRADUADOS

In [14]:
df_esp_matriculado = df_est[df_est['NIVEL_FORMACION']=='Especializacion']

In [15]:
df_grados_cod_uni['llave'] = df_grados_cod_uni['NUMERO'] + df_grados_cod_uni['COD_UNIDAD_MALLAS']
df_esp_matriculado['llave'] = df_esp_matriculado['NUMERO'] + df_esp_matriculado['COD_UNIDAD'] 

df_esp_matricula = df_esp_matriculado[~df_esp_matriculado['llave'].isin(df_grados_cod_uni['llave'])]

C:\Users\alejandro_arboledas\AppData\Local\Temp\ipykernel_20380\1010062227.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_esp_matriculado['llave'] = df_esp_matriculado['NUMERO'] + df_esp_matriculado['COD_UNIDAD']


In [16]:
df_esp_matricula[df_esp_matricula['NUMERO']=='1023891989']

,NUMERO,NIVEL_FORMACION,NOM_DEPENDENCIA,COD_UNIDAD,llave
167734,1023891989,Especializacion,ESPECIALIZACIÓN EN TRANSFORMACIÓN DIGITAL,ESP12,1023891989ESP12


In [17]:
def transformar_df(df):
    # Crear una columna con índice de dependencia para renombrar columnas correctamente
    df["NOM_DEPENDENCIA_INDEX"] = df.groupby("NUMERO").cumcount() + 1

    # Pivotear los valores de NOM_DEPENDENCIA
    df_pivot = df.pivot(index=["NUMERO", "NIVEL_FORMACION"], columns="NOM_DEPENDENCIA_INDEX", values="NOM_DEPENDENCIA")

    # Restablecer el índice para convertirlo en un DataFrame normal
    df_pivot.reset_index(inplace=True)

    # Renombrar las columnas de NOM_DEPENDENCIA
    df_pivot.columns = ["NUMERO", "NIVEL_FORMACION"] + [f"NOM_DEPENDENCIA_{i}" for i in range(1, len(df_pivot.columns) - 1)]

    return df_pivot

In [18]:
# Aplicar la función
df_transformado = transformar_df(df_esp_matricula)

C:\Users\alejandro_arboledas\AppData\Local\Temp\ipykernel_20380\3536986618.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["NOM_DEPENDENCIA_INDEX"] = df.groupby("NUMERO").cumcount() + 1


#### CRUZAR PARA TENER EN CUENTA LAS ESPECIALIZACIONES MATRICULADAS

In [19]:
df_final_especializaciones = pd.merge(df_grados_cod_uni,df_transformado,how='left',on=['NUMERO'])

In [20]:
df_final_especializaciones.rename(columns={'NOM_DEPENDENCIA_1':'ESP_MATRICULADO_1','NOM_DEPENDENCIA_2':'ESP_MATRICULADO_2'},inplace=True)

In [21]:
df_fin = df_final_especializaciones[['Nombres', 'genero_code', 'tipo_doc_unico', 'NUMERO', 'grad_annio',
       'grad_semestre', 'PLAN', 'snies', 'METODOLOGIA', 'NIVEL', 'TITULACION',
       'ESCUELA', 'fecha_grado', 'numero_de_credencial',
       'codigo_municipio_grado', 'SEDE_GRADO', 'acta', 'No_de_Registro',
       'folio', 'libro', 'ciclo_grado', 'Celular_Sugerido',
       'Telefono_Sugerido', 'Otros_Sugeridos', 'Correo_Personal_Sugerido',
       'PROGRAMA_PROFESIONAL', 'NOM_PROGRAMA_PROFESIONAL', 'SEDE_GRADO_2',
       'SCORE_M0', 'opcion_grado','NOM_DEPENDENCIA','ESP_MATRICULADO_1', 'ESP_MATRICULADO_2']]

df_fin[(df_fin['NOM_DEPENDENCIA'].notna())&(df_fin['ESP_MATRICULADO_1'].notna())&(df_fin['ESP_MATRICULADO_2'].notna())]

,Nombres,genero_code,tipo_doc_unico,NUMERO,grad_annio,grad_semestre,PLAN,snies,METODOLOGIA,NIVEL,...,Otros_Sugeridos,Correo_Personal_Sugerido,PROGRAMA_PROFESIONAL,NOM_PROGRAMA_PROFESIONAL,SEDE_GRADO_2,SCORE_M0,opcion_grado,NOM_DEPENDENCIA,ESP_MATRICULADO_1,ESP_MATRICULADO_2
21598,JULIAN ESTEBAN GARCIA FIGUEROA,o,CC,1026303837,2021,2,2024,52616,PRESENCIAL,PROFESIONAL,...,None,juliangarfigueroa17@gmail.com,INGENIERÍA DE SISTEMAS,INGENIERÍA DE SISTEMAS,"Bogotá, D.C.",No Contactado,HOMOLOGACION DIPLOMADO,ESPECIALIZACIÓN EN TRANSFORMACIÓN DIGITAL,ESPECIALIZACIÓN EN ANALÍTICA DE DATOS,ESPECIALIZACION EN GESTION DE TECNOLOGIAS DE L...
21599,JULIAN ESTEBAN GARCIA FIGUEROA,o,CC,1026303837,2021,2,2024,52616,PRESENCIAL,PROFESIONAL,...,None,juliangarfigueroa17@gmail.com,INGENIERÍA DE SISTEMAS,INGENIERÍA DE SISTEMAS,"Bogotá, D.C.",No Contactado,HOMOLOGACION DIPLOMADO,ESPECIALIZACIÓN EN TRANSFORMACIÓN DIGITAL,ESPECIALIZACIÓN EN ANALÍTICA DE DATOS,ESPECIALIZACION EN GESTION DE TECNOLOGIAS DE L...
41461,JOSE DAVID PUELLO AVILEZ,o,CC,1102862099,2020,2,2029,90688,PRESENCIAL,PROFESIONAL,...,None,josepuello13@gmail.com,INGENIERÍA DE SISTEMAS,INGENIERÍA DE SISTEMAS,Sincelejo,Contactado,CURSO NORMAL,ESPECIALIZACIÓN EN ANALÍTICA DE DATOS,ESPECIALIZACION EN GESTION DE TECNOLOGIAS DE L...,ESPECIALIZACIÓN EN TRANSFORMACIÓN DIGITAL
41462,JOSE DAVID PUELLO AVILEZ,o,CC,1102862099,2020,2,2029,90688,PRESENCIAL,PROFESIONAL,...,None,josepuello13@gmail.com,INGENIERÍA DE SISTEMAS,INGENIERÍA DE SISTEMAS,Sincelejo,Contactado,CURSO NORMAL,ESPECIALIZACIÓN EN ANALÍTICA DE DATOS,ESPECIALIZACION EN GESTION DE TECNOLOGIAS DE L...,ESPECIALIZACIÓN EN TRANSFORMACIÓN DIGITAL


### AHORA SE RASTREAN LOS DIPLOMADOS

In [22]:
df_diplomado = df_est[df_est['NIVEL_FORMACION']=='Diplomado']

In [23]:
df_diplomado = df_est[df_est['NIVEL_FORMACION']=='Diplomado']
df_diplomado[df_diplomado.duplicated(['NUMERO'])]

,NUMERO,NIVEL_FORMACION,NOM_DEPENDENCIA,COD_UNIDAD
299,1000004293,Diplomado,"DIPLOMADO GERENCIA, ADMINISTRACIÓN Y GESTIÓN D...",CE502
1197,1000046172,Diplomado,DIPLOMADO MACHINE LEARNING EN PYTHON,CE401
1279,1000047613,Diplomado,DIPLOMADO HABILIDADES GERENCIALES,CE503
1396,1000049402,Diplomado,DIPLOMADO EN COMMUNITY MANAGER,CE323
1635,1000064588,Diplomado,DIPLOMADO DESARROLLO DE APLICACIONES DE SOTFWA...,CE266
...,...,...,...,...
482417,98484782,Diplomado,DIPLOMADO SOSTENIBLE - UNA MIRADA AL SIGLO XXI,CE224
482576,98654644,Diplomado,DIPLOMADO GERENCIA DEL TALENTO HUMANO,CE277
482762,98763598,Diplomado,DIPLOMADO GERENCIA DEL TALENTO HUMANO,CE277
482763,98763598,Diplomado,DIPLOMADO GESTIÓN DE LA CALIDAD EN SALUD,CE235


In [24]:
df_diplomado = df_diplomado[['NUMERO','NOM_DEPENDENCIA']]

In [25]:
# Paso 1: Agrupar por cédula y consolidar los diplomados
df_diplomado_consolidado = (
    df_diplomado.groupby('NUMERO')
    .agg({
        'NOM_DEPENDENCIA': lambda x: ', '.join(sorted(x.dropna().unique())),  # Consolidar y ordenar diplomados
    })
    .reset_index()
)

# Paso 2: Separar los nombres de los diplomados en columnas individuales
max_diplomados = 3  # Cambia este valor si quieres manejar más diplomados
diplomado_columns = [f'DIPLOMADO_{i + 1}' for i in range(max_diplomados)]  # Crear nombres de columnas dinámicos

df_diplomado_consolidado[diplomado_columns] = (
    df_diplomado_consolidado['NOM_DEPENDENCIA']
    .str.split(', ', expand=True)  # Dividir en columnas
    .iloc[:, :max_diplomados]  # Limitar a max_diplomados
)

# Paso 3: Eliminar la columna consolidada si ya no es necesaria
df_diplomado_consolidado.drop(columns=['NOM_DEPENDENCIA'], inplace=True)

In [26]:
df_diplomado_consolidado[df_diplomado_consolidado['NUMERO']=='1000064588']

,NUMERO,DIPLOMADO_1,DIPLOMADO_2,DIPLOMADO_3
73,1000064588,DIPLOMADO EXCEL Y POWER BI,DIPLOMADO DESARROLLO DE APLICACIONES DE SOTFWA...,DIPLOMADO EN AUTOMATIZACIO´N DE PROCESOS INDUS...


### SE CRUZA PARA LLEVAR LOS DIPLOMADOS CURSADOS DE LOS GRADUADOS

In [27]:
df_esp_diplo = pd.merge(df_fin,df_diplomado_consolidado,how='left',on=['NUMERO'])

In [28]:
df_esp_diplo.shape

(83410, 36)

In [29]:
# Aplicar melt con las nuevas columnas agregadas
df_melted = df_esp_diplo.melt(
    id_vars=[
        'Nombres', 'genero_code', 'tipo_doc_unico', 'NUMERO', 'grad_annio',
        'grad_semestre', 'PLAN', 'snies', 'METODOLOGIA', 'NIVEL', 'TITULACION',
        'ESCUELA', 'fecha_grado', 'numero_de_credencial',
        'codigo_municipio_grado', 'SEDE_GRADO', 'acta', 'No_de_Registro',
        'folio', 'libro', 'ciclo_grado', 'Celular_Sugerido',
        'Telefono_Sugerido', 'Otros_Sugeridos', 'Correo_Personal_Sugerido',
        'PROGRAMA_PROFESIONAL', 'NOM_PROGRAMA_PROFESIONAL', 'SEDE_GRADO_2',
        'SCORE_M0', 'opcion_grado'
    ],
    value_vars=[
        'NOM_DEPENDENCIA', 'ESP_MATRICULADO_1', 'ESP_MATRICULADO_2',
        'DIPLOMADO_1', 'DIPLOMADO_2', 'DIPLOMADO_3'
    ],
    var_name='ORIGEN_PROGRAMA',
    value_name='PROGRAMA_RECOMPRA'
)

In [30]:
df_melted['NUMERO'].nunique()

82552

In [31]:
df_melted.shape

(500460, 32)

In [32]:
df_melted2 = df_melted[['Nombres', 'genero_code', 'tipo_doc_unico', 'NUMERO', 'grad_annio',
       'grad_semestre', 'PLAN', 'snies', 'METODOLOGIA', 'NIVEL', 'TITULACION',
       'ESCUELA', 'fecha_grado', 'numero_de_credencial',
       'codigo_municipio_grado', 'SEDE_GRADO', 'acta', 'No_de_Registro',
       'folio', 'libro', 'ciclo_grado', 'Celular_Sugerido',
       'Telefono_Sugerido', 'Otros_Sugeridos', 'Correo_Personal_Sugerido',
       'PROGRAMA_PROFESIONAL', 'NOM_PROGRAMA_PROFESIONAL', 'SEDE_GRADO_2',
       'SCORE_M0', 'opcion_grado', 'PROGRAMA_RECOMPRA']]

In [33]:
df_melted2 = df_melted2.drop_duplicates()

In [34]:
df_melted2['NUMERO'].nunique()

82552

In [35]:
df_melted2[df_melted2['NUMERO']=='1023891989']

,Nombres,genero_code,tipo_doc_unico,NUMERO,grad_annio,grad_semestre,PLAN,snies,METODOLOGIA,NIVEL,...,Celular_Sugerido,Telefono_Sugerido,Otros_Sugeridos,Correo_Personal_Sugerido,PROGRAMA_PROFESIONAL,NOM_PROGRAMA_PROFESIONAL,SEDE_GRADO_2,SCORE_M0,opcion_grado,PROGRAMA_RECOMPRA
18797,EDWIN ESTIL VALDERRAMA ARMERO,o,CC,1023891989,2018,2,1019,52605,PRESENCIAL,PROFESIONAL,...,3202248400,3717012.0,None,edwinvalderrama@hotmail.com,ADMINISTRACIÓN DE EMPRESAS,ADMINISTRACIÓN DE EMPRESAS,"Bogotá, D.C.",No Contactado,Homologación,ESPECIALIZACIÓN EN GERENCIA DE LA MARCA
18799,EDWIN ESTIL VALDERRAMA ARMERO,o,CC,1023891989,2018,2,1019,52605,PRESENCIAL,PROFESIONAL,...,3202248400,3717012.0,None,edwinvalderrama@hotmail.com,ADMINISTRACIÓN DE EMPRESAS,ADMINISTRACIÓN DE EMPRESAS,"Bogotá, D.C.",No Contactado,Homologación,ESPECIALIZACIÓN EN PAZ Y DESARROLLO TERRITORIAL
102207,EDWIN ESTIL VALDERRAMA ARMERO,o,CC,1023891989,2018,2,1019,52605,PRESENCIAL,PROFESIONAL,...,3202248400,3717012.0,None,edwinvalderrama@hotmail.com,ADMINISTRACIÓN DE EMPRESAS,ADMINISTRACIÓN DE EMPRESAS,"Bogotá, D.C.",No Contactado,Homologación,ESPECIALIZACIÓN EN TRANSFORMACIÓN DIGITAL
185617,EDWIN ESTIL VALDERRAMA ARMERO,o,CC,1023891989,2018,2,1019,52605,PRESENCIAL,PROFESIONAL,...,3202248400,3717012.0,None,edwinvalderrama@hotmail.com,ADMINISTRACIÓN DE EMPRESAS,ADMINISTRACIÓN DE EMPRESAS,"Bogotá, D.C.",No Contactado,Homologación,NaN


In [36]:
# Identificar grupos donde al menos un valor en PROGRAMA_RECOMPRA no es nulo
mask_valid_group = df_melted2.groupby('NUMERO')['PROGRAMA_RECOMPRA'].transform(lambda x: x.notna().any()).astype(bool)

# Crear una máscara para eliminar solo los valores vacíos dentro de los grupos válidos
mask_non_empty = df_melted2['PROGRAMA_RECOMPRA'].notna() & (df_melted2['PROGRAMA_RECOMPRA'] != "")

# Si un grupo tiene al menos un valor válido, aplicar el filtro; de lo contrario, dejar el grupo sin cambios
df_cleaned = df_melted2[~mask_valid_group | mask_non_empty]


In [37]:
df_cleaned[df_cleaned['PROGRAMA_RECOMPRA'].isnull()]
df_cleaned['PROGRAMA_RECOMPRA'].unique()

array([nan, 'ESPECIALIZACIÓN EN GERENCIA DE LA MARCA',
       'ESPECIALIZACIÓN EN ANALÍTICA DE DATOS',
       'ESPECIALIZACION EN GESTION DE TECNOLOGIAS DE LA INFORMACION',
       'ESPECIALIZACIÓN EN GESTIÓN DE TECNOLOGÍAS DE LA INFORMACIÓN',
       'ESPECIALIZACIÓN EN PAZ Y DESARROLLO TERRITORIAL',
       'ESPECIALIZACIÓN EN TRANSFORMACIÓN DIGITAL',
       'ESPECIALIZACIÓN EN GESTIÓN DE TECNOLOGÍAS DE INFORMACIÓN',
       'ESPECIALIZACIÓN EN GERENCIA DE LA INDUSTRIA EN LA MODA', 'NONE',
       'ESPECIALIZACIÓN EN GERENCIA FINANCIERA',
       'ESPECIALIZACIÓN EN MARKETING DIGITAL',
       'ESPECIALIZACIÓN EN GERENCIA DE PROYECTOS',
       'ESPECIALIZACIÓN EN ALTA GERENCIA',
       'ESPECIALIZACIÓN EN DESARROLLO ORGANIZACIONAL Y GESTIÓN DEL TALENTO HUMANO',
       'ESPECIALIZACIÓN EN INTELIGENCIA DE NEGOCIOS',
       'ESPECIALIZACIÓN EN GERENCIA EDUCATIVA',
       'ESPECIALIZACIÓN EN CONTRATACIÓN ESTATAL',
       'ESPECIALIZACIÓN EN PROSPECTIVA ESTRATÉGICA',
       'ESPECIALIZACION EN G

In [38]:
# Crear la nueva columna categorizando los valores de 'PROGRAMA_RECOMPRA'
df_cleaned['CATEGORIA_PROGRAMA'] = np.where(
    df_cleaned['PROGRAMA_RECOMPRA'].notna() & 
    (df_cleaned['PROGRAMA_RECOMPRA'].str.contains("ESPECIALIZACIÓN", case=False, na=False) |
     df_cleaned['PROGRAMA_RECOMPRA'].str.contains("ESPECIALIZACION", case=False, na=False)),
    "ESPECIALIZACION",
    np.where(df_cleaned['PROGRAMA_RECOMPRA'].isna() | (df_cleaned['PROGRAMA_RECOMPRA'] == ""),
             "NO RECOMPRA",
             "DIPLOMADOS"
    )
)

C:\Users\alejandro_arboledas\AppData\Local\Temp\ipykernel_20380\950360733.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cleaned['CATEGORIA_PROGRAMA'] = np.where(


In [39]:
df_cleaned.iloc[300:700:20]

a=df_cleaned[df_cleaned['CATEGORIA_PROGRAMA']=='ESPECIALIZACION']
a['PROGRAMA_RECOMPRA'].unique()

array(['ESPECIALIZACIÓN EN GERENCIA DE LA MARCA',
       'ESPECIALIZACIÓN EN ANALÍTICA DE DATOS',
       'ESPECIALIZACION EN GESTION DE TECNOLOGIAS DE LA INFORMACION',
       'ESPECIALIZACIÓN EN GESTIÓN DE TECNOLOGÍAS DE LA INFORMACIÓN',
       'ESPECIALIZACIÓN EN PAZ Y DESARROLLO TERRITORIAL',
       'ESPECIALIZACIÓN EN TRANSFORMACIÓN DIGITAL',
       'ESPECIALIZACIÓN EN GESTIÓN DE TECNOLOGÍAS DE INFORMACIÓN',
       'ESPECIALIZACIÓN EN GERENCIA DE LA INDUSTRIA EN LA MODA',
       'ESPECIALIZACIÓN EN GERENCIA FINANCIERA',
       'ESPECIALIZACIÓN EN MARKETING DIGITAL',
       'ESPECIALIZACIÓN EN GERENCIA DE PROYECTOS',
       'ESPECIALIZACIÓN EN ALTA GERENCIA',
       'ESPECIALIZACIÓN EN DESARROLLO ORGANIZACIONAL Y GESTIÓN DEL TALENTO HUMANO',
       'ESPECIALIZACIÓN EN INTELIGENCIA DE NEGOCIOS',
       'ESPECIALIZACIÓN EN GERENCIA EDUCATIVA',
       'ESPECIALIZACIÓN EN CONTRATACIÓN ESTATAL',
       'ESPECIALIZACIÓN EN PROSPECTIVA ESTRATÉGICA',
       'ESPECIALIZACION EN GESTION DE NEG

In [40]:
df_cleaned['NUMERO'].nunique()

82552

In [41]:
# Connection to database
host = '172.16.1.33'
port = '1433'
database = 'CUN_REPOSITORIO'
# username = 'swarm_cluster'  # Sustituye 'tu_usuario' por tu nombre de usuario real
# password = 'Repo2023*'  # Sustituye 'tu_contraseña' por tu contraseña real
#url = 'mssql+pyodbc://{username}:{password}@{host}:{port}/{db}?driver=ODBC Driver 17 for SQL Server'.format(username=username, password=password, host=host, port=port, db=database)
url = 'mssql+pyodbc://@{host}:{port}/{db}?driver=ODBC Driver 17 for SQL Server'.format(host=host, port=port, db=database)
engine = create_engine(url, fast_executemany=True)

table_name = 'RECOMPRA2'
schema_name = 'SRUBIO'

# Send the data to sql server
df_cleaned.to_sql(
    name=table_name,
    schema=schema_name, 
    con=engine,
    if_exists='replace',
    index=False,
    chunksize = 10000
)

# End connection
engine.dispose()

In [ ]:
from sqlalchemy import text
import pandas as pd

with engine.connect() as connection:
    try:
        # 1️⃣ Crear la tabla en SQL (si no existe) y cargarla
        create_table_query = text("""
        IF OBJECT_ID('##grados4', 'U') IS NOT NULL
            DROP TABLE ##grados4 ;

        SELECT numero
        into ##grados4 FROM GRADOS.Consolidado_Grados
        where NIVEL = 'profesional';
        """)

        connection.execute(create_table_query)
        connection.commit()

        # 2️⃣ Consultar la tabla ya creada y traerla a pandas
        select_query = text("""
        select c.* from ##grados4 as b
        inner join GRADOS.Consolidado_Grados as c
        on b.NUMERO = c.NUMERO
        where c.NIVEL = 'especializacion'
        """)

        df_especializacion_graduados = pd.read_sql_query(select_query, engine)

    finally:
        connection.close()


In [56]:
df_especializacion_graduados.shape

(852, 37)

In [57]:
# Connection to database
host = '172.16.1.33'
port = '1433'
database = 'CUN_REPOSITORIO'
# username = 'swarm_cluster'  # Sustituye 'tu_usuario' por tu nombre de usuario real
# password = 'Repo2023*'  # Sustituye 'tu_contraseña' por tu contraseña real
#url = 'mssql+pyodbc://{username}:{password}@{host}:{port}/{db}?driver=ODBC Driver 17 for SQL Server'.format(username=username, password=password, host=host, port=port, db=database)
url = 'mssql+pyodbc://@{host}:{port}/{db}?driver=ODBC Driver 17 for SQL Server'.format(host=host, port=port, db=database)
engine = create_engine(url, fast_executemany=True)

table_name = 'POST_GRADOSPROFESIONALES'
schema_name = 'GRADOS'

# Send the data to sql server
df_especializacion_graduados.to_sql(
    name=table_name,
    schema=schema_name, 
    con=engine,
    if_exists='replace',
    index=False,
    chunksize = 10000
)

# End connection
engine.dispose()